In [406]:
library(tidyverse)

In [407]:
library(readxl)
library(janitor)
library(stringr)

veg_surveys <- read_excel("datas_and_notebooks/2024/veg_surveys_digitalising.xlsx") |> 
  clean_names() |> 
  mutate(across(where(is.character), ~ str_trim(tolower(.))))

veg_surveys 

date,site,plot,subplot,diversity,id,survival,height,leaves,number_herbivory_damaged,hprop,number_fungal_pathogen_damaged,fprop,number_necrosis,nec_prop,notes
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
06.10.,a,ao22,ao22-i,1,646,na,26,12,12,2;40,na,na,na,na,NA
nv,a,ao22,ao22-ii,1,316,na,51,7,2,2;5,na,na,na,na,cf unlabled
nv,a,ao22,ao22-ii,1,623,NA,54,45,1,5;99,na,na,na,na,NA
nv,a,ao22,ao22-ii,1,1607,na,54,13,13,10;60,na,na,na,na,cf unlabled
nv,a,ao22,ao22-iii,1,696,na,62,113,47,2;49,na,na,na,na,4b
nv,a,ao22,ao22-iii,1,647,na,68,6,6,10;40,na,na,na,na,3b; dying
nv,a,ao27,ao27-ii,1,452,na,28,5,5,5;90,2,50;50,na,na,NA
nv,a,ao27,ao27-ii,1,133x,d,na,na,na,na,na,na,na,na,4b
nv,a,ao27,ao27-ii,1,397,na,na,na,na,na,na,na,na,na,NA


In [409]:
veg_surveys <- veg_surveys |> 
  mutate(across(where(is.character), ~ na_if(., "na"))) |> 
  mutate(
    id = if_else(id == "1205sapling", "1205", id),
    leaves = if_else(leaves == "2;3", "230", leaves),
    number_necrosis = if_else(str_detect(number_fungal_pathogen_damaged, "n"), 
                             number_fungal_pathogen_damaged, 
                             number_necrosis),
    nec_prop = if_else(str_detect(fprop, "n"), 
                      paste0(coalesce(nec_prop, ""), fprop), 
                      nec_prop),
    survival = case_when(
      survival == "seedling" ~ "s",
      survival == "woody recruit" ~ "r",
      survival %in% c("*", "27", "15") ~ NA_character_,
      TRUE ~ survival
    ),
    height = str_replace_all(height, ";", ".")
  ) |> 
  separate(number_herbivory_damaged, into = c("number_herbivory_min", "number_herbivory_max"), sep = ";", remove = FALSE) |>
  mutate(
    number_fungal_pathogen_damaged = if_else(str_detect(number_fungal_pathogen_damaged, "n"), NA_character_, number_fungal_pathogen_damaged),
    number_fungal_pathogen_damaged = str_replace_all(number_fungal_pathogen_damaged, ";f|\\(f\\)|\\(f|\\)|\\(|/\\(|f", ""),
    number_fungal_pathogen_damaged = case_when(
      str_detect(number_necrosis, "3f") ~ paste0(coalesce(number_fungal_pathogen_damaged, ""), "3"),
      str_detect(number_necrosis, "0f") ~ paste0(coalesce(number_fungal_pathogen_damaged, ""), "0"),
      TRUE ~ number_fungal_pathogen_damaged
    ),
    number_fungal_pathogen_damaged = case_when(
      number_fungal_pathogen_damaged == "/39" ~ "39",
      number_fungal_pathogen_damaged == "5;19" ~ "19",
      number_fungal_pathogen_damaged == "5;40" ~ "40",
      TRUE ~ number_fungal_pathogen_damaged
    )
  ) |>
  separate(hprop, into = c("hmin", "hmax"), sep = ";", remove = FALSE) |>
  mutate(
    hmin = if_else(hmin == "4(f)", "", hmin)
  )

Warning message:
“Expected 2 pieces. Missing pieces filled with `NA` in 219 rows [1, 2, 3, 4, 5,
6, 7, 12, 14, 15, 19, 20, 21, 22, 23, 28, 29, 30, 31, 33, ...].”
Warning message:
“Expected 2 pieces. Missing pieces filled with `NA` in 15 rows [21, 55, 59, 163,
189, 198, 274, 289, 299, 309, 520, 541, 558, 560, 583].”


In [410]:
veg_surveys <- veg_surveys |>
  dplyr::mutate(
    fprop = dplyr::case_when(
      nec_prop == "10;8(n)" ~ "10",
      nec_prop == "10f/20n" ~ "10",
      nec_prop == "5;10n" ~ "5",
      nec_prop == "40;90n" ~ "40",
      nec_prop == "10;40n" ~ "10",
      nec_prop == "5;15f/40n" ~ "5;15",
      nec_prop == "15;35f/20;30n" ~ "15;35",
      grepl("^5n$", nec_prop) ~ NA_character_,
      TRUE ~ fprop
    ),
    nec_prop = dplyr::case_when(
      nec_prop == "10;8(n)" ~ "8",
      nec_prop == "10f/20n" ~ "20",
      nec_prop == "5;10n" ~ "10",
      nec_prop == "40;90n" ~ "90",
      nec_prop == "10;40n" ~ "40",
      nec_prop == "5;15f/40n" ~ "40",
      nec_prop == "15;35f/20;30n" ~ "20",
      grepl("^5n$", nec_prop) ~ "5",
      grepl("^\\d+%n$", nec_prop) ~ sub("%n$", "", nec_prop),
      TRUE ~ nec_prop
    ),
    # Additional cleaning for fprop
    fprop = dplyr::case_when(
      fprop %in% c("25%n", "10%n") ~ NA_character_,
      grepl("/\\(5-7\\)", fprop) ~ "5;17",
      TRUE ~ fprop
    ),
    # Split fprop into fmin and fmax
    fmin = dplyr::case_when(
      !is.na(fprop) & grepl(";", fprop) ~ sub(";.*", "", fprop),
      !is.na(fprop) ~ fprop,
      TRUE ~ NA_character_
    ),
    fmax = dplyr::case_when(
      !is.na(fprop) & grepl(";", fprop) ~ sub(".*;", "", fprop),
      !is.na(fprop) & !grepl(";", fprop) ~ fprop,
      TRUE ~ NA_character_
    ),
    # Set number_fungal_pathogen_damaged to 15 if it is "10;28"
    number_fungal_pathogen_damaged = dplyr::case_when(
      number_fungal_pathogen_damaged == "10;28" ~ "15",
      TRUE ~ number_fungal_pathogen_damaged
    ),
    # Clean number_necrosis and number_fungal_pathogen_damaged based on rules
    number_necrosis = dplyr::case_when(
      number_necrosis %in% c("4(n)", "4n", "4") ~ "4",
      number_necrosis %in% c("25;n", "25n", "25") ~ "25",
      number_necrosis %in% c("12n", "12") ~ "12",
      number_necrosis %in% c("0f/5n", "Of/5n") ~ "5",
      number_necrosis %in% c("0f/4n", "Of/4n") ~ "4",
      number_necrosis %in% c("1n", "1") ~ "1",
      number_necrosis %in% c("17n", "17") ~ "17",
      number_necrosis %in% c("3f/1n") ~ "1",
      number_necrosis %in% c("3f/2n") ~ "2",
      number_necrosis %in% c("16/1n") ~ "1",
      number_necrosis %in% c("6(n)", "6n", "6") ~ "6",
      number_necrosis %in% c("2n", "2") ~ "2",
      number_necrosis %in% c("n1", "1n") ~ "1",
      number_necrosis %in% c("127n", "127") ~ "127",
      number_necrosis %in% c("3n", "3") ~ "3", number_necrosis %in% c("5n", 5) ~ "5", 
      TRUE ~ number_necrosis
    ),
    number_fungal_pathogen_damaged = dplyr::case_when(
      number_necrosis %in% c("0f/5n", "Of/5n") ~ "0",
      number_necrosis %in% c("0f/4n", "Of/4n") ~ "0",
      number_necrosis == "3f/1n" ~ "3",
      number_necrosis == "3f/2n" ~ "3",
      number_necrosis == "16/1n" ~ "16",
      TRUE ~ number_fungal_pathogen_damaged
    )
  ) |> 
 select(-c(number_herbivory_damaged, hprop, fprop, notes, date))

In [411]:
veg_surveys |> 
 str()

tibble [585 × 17] (S3: tbl_df/tbl/data.frame)
 $ site                          : chr [1:585] "a" "a" "a" "a" ...
 $ plot                          : chr [1:585] "ao22" "ao22" "ao22" "ao22" ...
 $ subplot                       : chr [1:585] "ao22-i" "ao22-ii" "ao22-ii" "ao22-ii" ...
 $ diversity                     : num [1:585] 1 1 1 1 1 1 1 1 1 1 ...
 $ id                            : chr [1:585] "646" "316" "623" "1607" ...
 $ survival                      : chr [1:585] NA NA NA NA ...
 $ height                        : chr [1:585] "26" "51" "54" "54" ...
 $ leaves                        : chr [1:585] "12" "7" "45" "13" ...
 $ number_herbivory_min          : chr [1:585] "12" "2" "1" "13" ...
 $ number_herbivory_max          : chr [1:585] NA NA NA NA ...
 $ hmin                          : chr [1:585] "2" "2" "5" "10" ...
 $ hmax                          : chr [1:585] "40" "5" "99" "60" ...
 $ number_fungal_pathogen_damaged: chr [1:585] NA NA NA NA ...
 $ number_necrosis               :

In [412]:
sapling_survey_2024 <- read_excel("datas_and_notebooks/2024/Sapling survey 2024.xlsx") |> 
  janitor::clean_names() |> 
  mutate(across(everything(), ~ tolower(str_trim(.)))) |> 
  mutate(across(everything(), ~ ifelse(. == "na", NA, .)))
sapling_survey_2024 |> 
  str()

Warning message:
“Coercing text to numeric in E1237 / R1237C5: '909'”
Warning message:
“Coercing text to numeric in E1238 / R1238C5: '898'”
Warning message:
“Coercing text to numeric in E1241 / R1241C5: '744'”
Warning message:
“Coercing text to numeric in E1242 / R1242C5: '1522'”
Warning message:
“Coercing text to numeric in E1244 / R1244C5: '865'”
Warning message:
“Coercing text to numeric in E1245 / R1245C5: '226'”
New names:
• `` -> `...8`


tibble [1,496 × 24] (S3: tbl_df/tbl/data.frame)
 $ date                          : chr [1:1496] "45438" "45438" "45438" "45438" ...
 $ site                          : chr [1:1496] "a" "a" "a" "a" ...
 $ plot                          : chr [1:1496] "ac32" "ac32" "ac32" "ac32" ...
 $ subplot                       : chr [1:1496] "ac32-i" "ac32-i" "ac32-i" "ac32-i" ...
 $ diversity                     : chr [1:1496] "2" "2" "2" NA ...
 $ id                            : chr [1:1496] "276" "327" "597" "1658" ...
 $ species                       : chr [1:1496] "lindera reflexa" "lindera reflexa" "lindera reflexa" "seedling" ...
 $ x8                            : chr [1:1496] "v" "v" "v" "v" ...
 $ survival                      : chr [1:1496] "a" "1" "1" "s" ...
 $ height                        : chr [1:1496] NA "70" "86" "6" ...
 $ leaves                        : chr [1:1496] NA "93" "250" "6" ...
 $ number_herbivory_damaged      : chr [1:1496] NA "14" "0" "3" ...
 $ hprop                    

In [413]:
(surveys_24 <- sapling_survey_2024 |> 
 #distinct(id) |> 
 full_join(veg_surveys |> 
		   mutate(diversity = as.character(diversity))) |> 
  select(id, everything()))

Joining with `by = join_by(site, plot, subplot, diversity, id, survival,
height, leaves, number_fungal_pathogen_damaged, number_necrosis, nec_prop)`


id,date,site,plot,subplot,diversity,species,x8,survival,height,⋯,nec_prop,percent_nmin,percent_nmax,notes,number_herbivory_min,number_herbivory_max,hmin,hmax,fmin,fmax
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
276,45438,a,ac32,ac32-i,2,lindera reflexa,v,a,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
327,45438,a,ac32,ac32-i,2,lindera reflexa,v,1,70,⋯,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
597,45438,a,ac32,ac32-i,2,lindera reflexa,v,1,86,⋯,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
1658,45438,a,ac32,ac32-i,NA,seedling,v,s,6,⋯,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
1520,45438,a,ac32,ac32-i,NA,"quercus sp,",v,n,52,⋯,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
1526,45438,a,ac32,ac32-i,NA,seedling,v,s,5,⋯,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
666,45438,a,ac32,ac32-ii,2,quercus fabri,v,1,44,⋯,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
333,45438,a,ac32,ac32-ii,2,syzygium buxifolium,v,1,86,⋯,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
660,45438,a,ac32,ac32-iii,2,adinandra millettii,v,1,120,⋯,0,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [414]:
final_joined <- read.csv('datas_and_notebooks/final_datas/final_joined.csv') |> 
 select(-X)
final_joined |> 
 str()

'data.frame':	2160 obs. of  42 variables:
 $ date                   : chr  "2023-09-07" "2023-09-07" "2023-09-07" "2023-09-07" ...
 $ site                   : chr  "a" "a" "a" "a" ...
 $ plot                   : chr  "ae31" "ae31" "ae31" "ae31" ...
 $ subplot                : chr  "ae31-i" "ae31-i" "ae31-ii" "ae31-iii" ...
 $ diversity              : int  1 1 1 1 1 1 1 1 1 1 ...
 $ id                     : chr  "340" "92" "250" "248" ...
 $ species                : chr  "quercus serrata" "quercus serrata" "dalbergia hupeana" "rhododendron simsii" ...
 $ genus                  : chr  "quercus" "quercus" "dalbergia" "rhododendron" ...
 $ family                 : chr  "fagaceae" "fagaceae" "fabaceae" "ericaceae" ...
 $ growth_form            : chr  "tree" "tree" "tree" "shrub/tree" ...
 $ seed_source_short      : chr  "II" "II" "EI" "ASP" ...
 $ type                   : chr  NA "sapling" "sapling" "sapling" ...
 $ survival               : chr  "d" "1" "1" "1" ...
 $ height                

In [416]:
library(dplyr)
library(tidyr)
library(stringr)
library(purrr)

surveys_24_updated <- surveys_24 |> 
  left_join(final_joined |> distinct(id, plot, subplot, species2 = species, diversity2 = diversity, family, genus, growth_form, seed_source_short )) |> 
  mutate(
    species = if_else(is.na(species) & !is.na(species2), species2, species)
  ) |> 
  select(-species2, x8) |> 
  filter(!is.na(id)) |> 
  mutate(
    number_necrosis = stringr::str_replace(number_necrosis, "/", ""),
    number_fungal_pathogen_damaged = stringr::str_replace(number_fungal_pathogen_damaged, "0/", "0"), 
    subplot = stringr::str_replace(subplot, "bu16- iv", "bu16-iv"),
    diversity = stringr::str_replace_all(diversity, "226", "1"), 
    subplot = stringr::str_replace_all(subplot, "ar33.ii", "ae33.ii")
  ) |> 
  mutate(
    number_herbivory_min = if_else(
      is.na(number_herbivory_min) & !is.na(hmin), hmin, number_herbivory_min
    ),
    number_herbivory_max = if_else(
      is.na(number_herbivory_max) & !is.na(hmax), hmax, number_herbivory_max
    )
  ) |> 
  select(-c(hmax, hmin)) |> 
  group_by(plot, subplot) |> 
  mutate(
    diversity2 = if_else(
      is.na(diversity2) & any(!is.na(diversity2)), 
      unique(na.omit(diversity2))[1], 
      diversity2
    )
  ) |> 
  mutate(
    diversity2 = if_else(
      is.na(diversity2) & !is.na(diversity), as.character(diversity), as.character(diversity2)
    )
  ) |> 
  ungroup() |> 
  mutate(
    diversity2 = if_else(
      stringr::str_detect(subplot, "ae33.ii|ai28.iv"), "1", diversity2
    )
  ) |> 
  select(date, plot, subplot, id, species, everything(), -diversity, -x8) |> 
  rename("diversity" = diversity2) |> 
  mutate(
    height = as.numeric(height),
    leaves = as.numeric(leaves),
    number_herbivory_damaged = as.numeric(number_herbivory_damaged),
    hprop = as.numeric(hprop),
    percent_hmin = as.numeric(percent_hmin),
    percent_hmax = as.numeric(percent_hmax),
    number_fungal_pathogen_damaged = as.numeric(number_fungal_pathogen_damaged),
    fprop = as.numeric(fprop),
    percent_fmin = as.numeric(percent_fmin),
    percent_fmax = as.numeric(percent_fmax),
    number_necrosis = as.numeric(number_necrosis),
    nec_prop = as.numeric(nec_prop),
    percent_nmin = as.numeric(percent_nmin),
    percent_nmax = as.numeric(percent_nmax),
    number_herbivory_min = as.numeric(number_herbivory_min),
    number_herbivory_max = as.numeric(number_herbivory_max),
    fmin = as.numeric(fmin),
    fmax = as.numeric(fmax)
  )

# Define a named vector of corrections: wrong = right
species_corrections <- c(
  "camellia sinesis" = "camellia sinensis",
  "camellia sp," = "camellia sp.",
  "quercus sp," = "quercus sp.",
  "seedming" = "seedling",
  "seedlig" = "seedling",
  "machilus grijsi" = "machilus grijesi",
  "machilus grijesi'" = "machilus grijesi",
  "lindera reflexa'" = "lindera reflexa",
  "camellia fra'trema cannabina" = "camellia fra'trema cannabina", # unclear, left as is
  "herb'" = "herb",
  "rosa sp.'" = "rosa sp.",
  "camellia sp.''" = "camellia sp.",
  "camellia sp." = "camellia sp.",
  "quercus seedling" = "seedling",
  "woody recruit" = "recruit",
  "seedling (cfr. litsea)" = "seedling"
)

# Clean up species names
surveys_24_updated <- surveys_24_updated %>%
  mutate(
    species = str_trim(species),
    species = str_to_lower(species),
    species = str_replace_all(species, "[‘’'`]", ""), # remove stray quotes
    species = str_replace_all(species, "\\s+", " "),  # collapse multiple spaces
    species = str_replace(species, ",$", "."),        # replace trailing comma with period
    species = if_else(species %in% names(species_corrections), species_corrections[species], species),
    species = if_else(species %in% c("na", "", "unknown sp.", "look at photo", "not-liquidambar", "444", "1171", "307"), NA_character_, species)
  )

surveys_24_updated |> 
  str()

Joining with `by = join_by(id, plot, subplot)`
Warning message in left_join(surveys_24, distinct(final_joined, id, plot, subplot, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 512 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message:
“There were 10 warnings in `mutate()`.
The first warning was:
ℹ In argument: `height = as.numeric(height)`.
Caused by warning:
! NAs introduced by coercion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 9 remaining warnings.”


tibble [2,646 × 31] (S3: tbl_df/tbl/data.frame)
 $ date                          : chr [1:2646] "45438" "45438" "45438" "45438" ...
 $ plot                          : chr [1:2646] "ac32" "ac32" "ac32" "ac32" ...
 $ subplot                       : chr [1:2646] "ac32-i" "ac32-i" "ac32-i" "ac32-i" ...
 $ id                            : chr [1:2646] "276" "276" "327" "327" ...
 $ species                       : Named chr [1:2646] "lindera reflexa" "lindera reflexa" "lindera reflexa" "lindera reflexa" ...
  ..- attr(*, "names")= chr [1:2646] "" "" "" "" ...
 $ site                          : chr [1:2646] "a" "a" "a" "a" ...
 $ survival                      : chr [1:2646] "a" "a" "1" "1" ...
 $ height                        : num [1:2646] NA NA 70 70 86 86 6 52 5 44 ...
 $ leaves                        : num [1:2646] NA NA 93 93 250 250 6 17 2 59 ...
 $ number_herbivory_damaged      : num [1:2646] NA NA 14 14 0 0 3 12 0 24 ...
 $ hprop                         : num [1:2646] NA NA 0.151 0.151

In [417]:
surveys_24_updated <- surveys_24_updated |>
  rename(
    "number_leaves" = leaves,
    "number_fungal_damaged" = number_fungal_pathogen_damaged,
    "number_necrotic_damaged" = number_necrosis,
    "percent_necrose" = nec_prop,
	  "number_herb_damaged" = number_herbivory_damaged,
    "percent_fungal" = fprop
  ) |>
  mutate(
    percent_hmin = ifelse(
      is.na(percent_hmin) & !is.na(number_herbivory_min) & !is.na(number_leaves),
      as.numeric(number_herbivory_min) / as.numeric(number_leaves),
      as.numeric(percent_hmin)
    ),
    percent_hmax = ifelse(
      is.na(percent_hmax) & !is.na(number_herbivory_max) & !is.na(number_leaves),
      as.numeric(number_herbivory_max) / as.numeric(number_leaves),
      as.numeric(percent_hmax)
    ),
    percent_fmin = ifelse(
      is.na(percent_fmin) & !is.na(fmin) & !is.na(number_leaves),
      as.numeric(fmin) / as.numeric(number_leaves),
      as.numeric(percent_fmin)
    ),
    percent_fmax = ifelse(
      is.na(percent_fmax) & !is.na(fmax) & !is.na(number_leaves),
      as.numeric(fmax) / as.numeric(number_leaves),
      as.numeric(percent_fmax)
    )
  ) |> 
 select(-c(fmin, fmax, hprop)) |> 
 mutate(year = as.integer(paste("2024")))
surveys_24_updated |> 
 str()

tibble [2,646 × 29] (S3: tbl_df/tbl/data.frame)
 $ date                   : chr [1:2646] "45438" "45438" "45438" "45438" ...
 $ plot                   : chr [1:2646] "ac32" "ac32" "ac32" "ac32" ...
 $ subplot                : chr [1:2646] "ac32-i" "ac32-i" "ac32-i" "ac32-i" ...
 $ id                     : chr [1:2646] "276" "276" "327" "327" ...
 $ species                : Named chr [1:2646] "lindera reflexa" "lindera reflexa" "lindera reflexa" "lindera reflexa" ...
  ..- attr(*, "names")= chr [1:2646] "" "" "" "" ...
 $ site                   : chr [1:2646] "a" "a" "a" "a" ...
 $ survival               : chr [1:2646] "a" "a" "1" "1" ...
 $ height                 : num [1:2646] NA NA 70 70 86 86 6 52 5 44 ...
 $ number_leaves          : num [1:2646] NA NA 93 93 250 250 6 17 2 59 ...
 $ number_herb_damaged    : num [1:2646] NA NA 14 14 0 0 3 12 0 24 ...
 $ percent_hmin           : num [1:2646] NA NA 0.01 0.01 NA NA 0.2 0.05 NA 0.01 ...
 $ percent_hmax           : num [1:2646] NA NA 0.5 

In [418]:
library(readxl)
library(stringr)
library(dplyr)

surveys_all <- read_excel("datas_and_notebooks/2024/Vegetation survey spring 2024 (1 subplot missing).xlsx") |> 
  janitor::clean_names() |> 
  mutate(across(everything(), ~ if(is.character(.)) str_trim(tolower(.)) else .)) |>
  select(-contains("spalte")) |>
  rename(
    "herb_max" = max_herb_layer_height,
    "deadwood" = deadwppd, 
	  "woody" = "woody_plants", 
	  "leaflitter_dept" = leaf_litter_dept, 
	  "leaflitter" = leaf_litter, 
	  "herbaceous" = herb
  ) |>
  mutate(across(4:15, as.numeric)) |> 
  full_join(
    surveys_24_updated |> 
      rename("notes2" = notes) |>  
      select(-date) |> 
      mutate(diversity = as.integer(diversity))
  ) |> 
  distinct() |> 
  select(    date, site, plot, subplot, diversity, id, species, genus, family, growth_form, seed_source_short,
    everything()
  ) |> 
				select(-c(number_herbivory_min, number_herbivory_max)) |> 
				select(everything(), notes, notes2) |> 
				full_join(final_joined |> 
						 mutate(date = as.Date(date))) 

surveys_all |> 
  str()

Warning message:
“There were 12 warnings in `mutate()`.
The first warning was:
ℹ In argument: `across(4:15, as.numeric)`.
Caused by warning:
! NAs introduced by coercion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 11 remaining warnings.”
Joining with `by = join_by(plot, subplot)`
Warning message in full_join(mutate(rename(select(mutate(janitor::clean_names(read_excel("datas_and_notebooks/2024/Vegetation survey spring 2024 (1 subplot missing).xlsx")), :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 478 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Joining with `by = join_by(date, site, plot, subplot, diversity, id, species,
genus, family, growth_form, seed_source_short, tot_veg, herb_max, grass, fern,
herbaceous, woody, rock, leaflitter, deadwood, moss, notes, survival, height,
number_leaves, number_herb_dama

tibble [4,833 × 45] (S3: tbl_df/tbl/data.frame)
 $ date                   : POSIXct[1:4833], format: "2024-05-26" "2024-05-26" ...
 $ site                   : chr [1:4833] "a" "a" "a" "a" ...
 $ plot                   : chr [1:4833] "ac32" "ac32" "ac32" "ac32" ...
 $ subplot                : chr [1:4833] "ac32-i" "ac32-i" "ac32-i" "ac32-i" ...
 $ diversity              : int [1:4833] 2 2 2 2 2 2 2 2 2 2 ...
 $ id                     : chr [1:4833] "276" "276" "327" "327" ...
 $ species                : Named chr [1:4833] "lindera reflexa" "lindera reflexa" "lindera reflexa" "lindera reflexa" ...
  ..- attr(*, "names")= chr [1:4833] "" "" "" "" ...
 $ genus                  : chr [1:4833] "lindera" "lindera" "lindera" "lindera" ...
 $ family                 : chr [1:4833] "lauraceae" "lauraceae" "lauraceae" "lauraceae" ...
 $ growth_form            : chr [1:4833] "shrub/tree" "shrub/tree" "shrub/tree" "shrub/tree" ...
 $ seed_source_short      : chr [1:4833] "EI" NA "EI" NA ...
 $ tot_v

In [419]:
surveys_all |> 
 write.csv("datas_and_notebooks/final_datas/all_data_joined.csv", row.names = F) 

In [420]:
final_joined |> 
  str()

'data.frame':	2160 obs. of  42 variables:
 $ date                   : chr  "2023-09-07" "2023-09-07" "2023-09-07" "2023-09-07" ...
 $ site                   : chr  "a" "a" "a" "a" ...
 $ plot                   : chr  "ae31" "ae31" "ae31" "ae31" ...
 $ subplot                : chr  "ae31-i" "ae31-i" "ae31-ii" "ae31-iii" ...
 $ diversity              : int  1 1 1 1 1 1 1 1 1 1 ...
 $ id                     : chr  "340" "92" "250" "248" ...
 $ species                : chr  "quercus serrata" "quercus serrata" "dalbergia hupeana" "rhododendron simsii" ...
 $ genus                  : chr  "quercus" "quercus" "dalbergia" "rhododendron" ...
 $ family                 : chr  "fagaceae" "fagaceae" "fabaceae" "ericaceae" ...
 $ growth_form            : chr  "tree" "tree" "tree" "shrub/tree" ...
 $ seed_source_short      : chr  "II" "II" "EI" "ASP" ...
 $ type                   : chr  NA "sapling" "sapling" "sapling" ...
 $ survival               : chr  "d" "1" "1" "1" ...
 $ height                

In [421]:
# Corrected code
data.frame(Column_Names = colnames(final_joined))

Column_Names
<chr>
date
site
plot
subplot
diversity
id
species
genus
family


In [422]:
library(dplyr)
library(stringr)
library(purrr)


# Show unique species after cleaning
unique_species <- surveys_24_updated %>%
  distinct(species) %>%
  arrange(species)

unique_species

species
<chr>
actinidia melanandra
actinidia polygama
adinandra millettii
ailanthus altissima
albizia julibrissin
alniphyllum fortunei
aralia chinensis
ardisia crenata
betula luminifera
